# AGROBRIDGE — Pomegranate Dataset Preparation Pipeline

**Objective**: Ingest audited pomegranate fruit disease images, map folder labels to the standard 5-class taxonomy, filter out duplicates & corrupt images, and generate leakage-safe stratified `train`, `val`, and `test` manifests.

> **Data Policy**: Cloud runtime execution only (Kaggle / Colab). Datasets live in ephemeral cloud storage (`/kaggle/input` or `/content/drive`). Never commit raw image binaries to Git.

In [ ]:
import os
import json
import hashlib
from pathlib import Path
from collections import Counter
from PIL import Image
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Set seeds for reproducible splitting
SEED = 42
np.random.seed(SEED)

# Cloud environment paths
KAGGLE_INPUT = Path('/kaggle/input')
COLAB_INPUT = Path('/content/pomegranate_data')
WORKING_DIR = Path('/kaggle/working/pomegranate_prepared') if KAGGLE_INPUT.exists() else Path('./pomegranate_prepared')
WORKING_DIR.mkdir(parents=True, exist_ok=True)

print(f"Working output directory: {WORKING_DIR.resolve()}")

## 1. Class Taxonomy Mapping
Normalize diverse folder names from public datasets (Kaggle/Mendeley) into the canonical 5 classes.

In [ ]:
CANONICAL_CLASSES = ["Healthy", "Anthracnose", "Bacterial_Blight", "Alternaria", "Cercospora"]

LABEL_MAP = {
    # Healthy variations
    "healthy": "Healthy",
    "healthy_fruit": "Healthy",
    "healthy_pomegranate": "Healthy",
    # Anthracnose variations
    "anthracnose": "Anthracnose",
    "anthracnose_fruit": "Anthracnose",
    "colletotrichum": "Anthracnose",
    # Bacterial blight variations
    "bacterial_blight": "Bacterial_Blight",
    "bacterial blight": "Bacterial_Blight",
    "telya": "Bacterial_Blight",
    "xanthomonas": "Bacterial_Blight",
    # Alternaria variations
    "alternaria": "Alternaria",
    "alternaria_black_spot": "Alternaria",
    "heart_rot": "Alternaria",
    # Cercospora variations
    "cercospora": "Cercospora",
    "cercospora_fruit_spot": "Cercospora",
    "cercospora_spot": "Cercospora",
}

def normalize_label(dir_name: str):
    cleaned = dir_name.strip().lower().replace(" ", "_").replace("-", "_")
    for pattern, target in LABEL_MAP.items():
        if pattern in cleaned:
            return target
    return None

## 2. Ingestion, Verification & Deduplication

In [ ]:
def scan_and_audit_dataset(dataset_dir: Path):
    valid_extensions = {".jpg", ".jpeg", ".png", ".webp"}
    records = []
    hashes = {}
    corrupt_count = 0
    dup_count = 0

    for root, dirs, files in os.walk(dataset_dir):
        folder_name = Path(root).name
        canonical_label = normalize_label(folder_name)
        if not canonical_label:
            continue

        for fname in files:
            p = Path(root) / fname
            if p.suffix.lower() not in valid_extensions:
                continue

            # 1. Check for corrupt image
            try:
                with Image.open(p) as img:
                    img.verify()
                with Image.open(p) as img:
                    w, h = img.size
                    mode = img.mode
            except Exception as e:
                corrupt_count += 1
                continue

            # 2. Check for exact duplicate via SHA256
            try:
                file_hash = hashlib.sha256(p.read_bytes()).hexdigest()
                if file_hash in hashes:
                    dup_count += 1
                    continue
                hashes[file_hash] = str(p)
            except Exception:
                continue

            records.append({
                "path": str(p.resolve()),
                "filename": fname,
                "label": canonical_label,
                "width": w,
                "height": h,
                "mode": mode,
                "sha256": file_hash
            })

    print(f"Audit Complete! Found {len(records)} clean images. Corrupt: {corrupt_count}, Exact Duplicates Dropped: {dup_count}")
    return pd.DataFrame(records)

# Locate dataset directory
dataset_input_dir = None
if KAGGLE_INPUT.exists():
    candidates = list(KAGGLE_INPUT.iterdir())
    if candidates:
        dataset_input_dir = candidates[0]
elif COLAB_INPUT.exists():
    dataset_input_dir = COLAB_INPUT

if dataset_input_dir and dataset_input_dir.exists():
    print(f"Scanning dataset from: {dataset_input_dir}")
    df = scan_and_audit_dataset(dataset_input_dir)
else:
    print("Demo/dry-run mode: Mocking clean dataset entries for pipeline verification...")
    sample_records = []
    for cls in CANONICAL_CLASSES:
        for i in range(120):
            sample_records.append({
                "path": f"/mock/data/{cls}/img_{i:04d}.jpg",
                "filename": f"img_{i:04d}.jpg",
                "label": cls,
                "width": 512,
                "height": 512,
                "mode": "RGB",
                "sha256": hashlib.sha256(f"{cls}_{i}".encode()).hexdigest()
            })
    df = pd.DataFrame(sample_records)

print("\nClass counts:")
print(df["label"].value_counts())

## 3. Leakage-Safe Stratified Splitting (70% Train, 15% Val, 15% Test)

In [ ]:
# Split: 70% train, 30% temp
train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=SEED, stratify=df["label"]
)

# Split temp equally: 15% val, 15% test
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["label"]
)

# STRICT LEAKAGE AUDIT: Verify 0 overlapping paths or SHA hashes
train_set = set(train_df["sha256"])
val_set = set(val_df["sha256"])
test_set = set(test_df["sha256"])

assert len(train_set & val_set) == 0, "Data leakage detected between train and val!"
assert len(train_set & test_set) == 0, "Data leakage detected between train and test!"
assert len(val_set & test_set) == 0, "Data leakage detected between val and test!"
print("✓ Zero-leakage verification PASSED! Splits are mutually exclusive.")

print(f"Train samples: {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)")
print(f"Validation samples: {len(val_df)} ({len(val_df)/len(df)*100:.1f}%)")
print(f"Test samples: {len(test_df)} ({len(test_df)/len(df)*100:.1f}%)")

## 4. Export Manifests and Taxonomy Summary

In [ ]:
# Save clean JSON and CSV manifests
train_df.to_csv(WORKING_DIR / "train_manifest.csv", index=False)
val_df.to_csv(WORKING_DIR / "val_manifest.csv", index=False)
test_df.to_csv(WORKING_DIR / "test_manifest.csv", index=False)

train_df.to_json(WORKING_DIR / "train_manifest.json", orient="records", indent=2)
val_df.to_json(WORKING_DIR / "val_manifest.json", orient="records", indent=2)
test_df.to_json(WORKING_DIR / "test_manifest.json", orient="records", indent=2)

manifest_summary = {
    "canonical_classes": CANONICAL_CLASSES,
    "num_classes": len(CANONICAL_CLASSES),
    "seed": SEED,
    "total_clean_samples": len(df),
    "train_samples": len(train_df),
    "val_samples": len(val_df),
    "test_samples": len(test_df),
    "class_distribution": {
        "train": train_df["label"].value_counts().to_dict(),
        "val": val_df["label"].value_counts().to_dict(),
        "test": test_df["label"].value_counts().to_dict()
    }
}

with open(WORKING_DIR / "dataset_summary.json", "w") as f:
    json.dump(manifest_summary, f, indent=2)

print("\nManifests successfully generated and saved to:", WORKING_DIR)